In [ ]:
# To Do: AI Use Disclaimer

In [13]:
import pandas as pd
import duckdb
import polars as pl

In [40]:
# TO DO - Load new file with company response filter
# TO DO - Drop Sub-Product and Sub-Issue nulls

# Load parquet into lazy dataframe and display the head
PARQUET_PATH = '../data/processed/consumer_banking_complaints.parquet'
df = pl.scan_parquet(PARQUET_PATH)

print(f"Total number of records: {df.select(pl.len()).collect()}")
df.head(5).collect()

Total number of records: shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 1134035 │
└─────────┘


Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
date,str,str,str,str,str,str,str,str,str,str,str,str,date,str,bool,str,i64
2019-12-26,"""Credit card or prepaid card""","""General-purpose credit card or…","""Advertising and marketing, inc…","""Confusing or misleading advert…",null,null,"""CAPITAL ONE FINANCIAL CORPORAT…","""CA""","""94025""",null,"""Consent not provided""","""Web""",2019-12-26,"""Closed with explanation""",true,"""N/A""",3477549
2019-12-20,"""Checking or savings account""","""Other banking product or servi…","""Managing an account""","""Funds not handled or disbursed…",null,"""Company has responded to the c…","""WELLS FARGO & COMPANY""","""FL""","""33064""",null,"""N/A""","""Referral""",2019-12-23,"""Closed with explanation""",true,"""N/A""",3475858
2019-11-18,"""Credit card or prepaid card""","""General-purpose credit card or…","""Problem with a purchase shown …","""Credit card company isn't reso…","""XXXX claimed they delivered a …",null,"""DISCOVER BANK""","""MA""","""021XX""",null,"""Consent provided""","""Web""",2019-11-18,"""Closed with explanation""",true,"""N/A""",3442136
2020-06-05,"""Checking or savings account""","""Checking account""","""Managing an account""","""Problem using a debit or ATM c…",null,"""Company has responded to the c…","""CITIBANK, N.A.""","""NY""","""10466""",null,"""Consent not provided""","""Web""",2020-06-05,"""Closed with explanation""",true,"""N/A""",3684669
2024-01-16,"""Credit card""","""General-purpose credit card or…","""Other features, terms, or prob…","""Add-on products and services""",null,"""Company has responded to the c…","""WELLS FARGO & COMPANY""","""TX""","""76179""",null,"""Consent not provided""","""Web""",2024-01-16,"""Closed with monetary relief""",true,"""N/A""",8161600


In [15]:
# # Drop null/blank narratives
#
# narratives_df = (
#     df
#     .filter(
#         pl.col('Consumer complaint narrative').is_not_null()
#         & (pl.col('Consumer complaint narrative').str.strip_chars() != '')
#     )
#     .collect()
#     .to_pandas()
# )

In [41]:
# Add word counts to the lazyframe
# Blanks/nulls get a word count of 0
df = (
    df
    .with_columns(
        pl.col('Consumer complaint narrative')
        .fill_null('')
        .str.extract_all(r'\b\w+\b')
        .list.len()
        .alias('word_count')
    )
)

In [46]:
row_count = (
    df
    .select(pl.len())
    .collect()
    .item()
)

print(row_count)

1134035


In [47]:
non_null_counts = (
    df
    .select(pl.all().count())
    .collect()
)

non_null_counts

Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1134035,1134035,1105736,1134030,869805,541497,562889,1134035,1114965,1120163,201643,1084329,1134035,1134035,1134034,1134035,1134035,1134035,1134035


In [17]:
# Replace blank/null narratives with '[No Narrative]'
df_cleaned = (
    df
    .with_columns(
        pl.when(
            pl.col('Consumer complaint narrative').is_null() |
            (pl.col('Consumer complaint narrative').str.strip_chars() == '')
        )
        .then(pl.lit('[No Narrative]'))
        .otherwise(pl.col('Consumer complaint narrative'))
        .alias('Consumer complaint narrative')
    )
)

In [18]:
df_pandas = (
    df_cleaned
    .filter(pl.col('Consumer complaint narrative') != '[No Narrative]')
    .collect()
    .to_pandas()
)

In [19]:
df_pandas.info()

<class 'pandas.DataFrame'>
RangeIndex: 541497 entries, 0 to 541496
Data columns (total 19 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   Date received                 541497 non-null  datetime64[ms]
 1   Product                       541497 non-null  str           
 2   Sub-product                   528868 non-null  str           
 3   Issue                         541497 non-null  str           
 4   Sub-issue                     422230 non-null  str           
 5   Consumer complaint narrative  541497 non-null  str           
 6   Company public response       266484 non-null  str           
 7   Company                       541497 non-null  str           
 8   State                         538905 non-null  str           
 9   ZIP code                      541497 non-null  str           
 10  Tags                          114209 non-null  str           
 11  Consumer consent provide

In [32]:
# Check for duplicate Complaint ID
duplicate_row_count = df_pandas['Complaint ID'].duplicated(keep=False).sum()
duplicate_row_count

np.int64(0)

In [33]:
# Check for duplicate narratives
duplicate_row_count = df_pandas['Consumer complaint narrative'].duplicated(keep=False).sum()
duplicate_row_count

np.int64(20540)

In [34]:
# Inspect the rows
duplicate_rows_df = (
    df_pandas
    .loc[df_pandas['Consumer complaint narrative'].duplicated(keep=False)]
    .sort_values('Consumer complaint narrative')
)

duplicate_rows_df

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count
129918,2023-11-20,Credit card,General-purpose credit card or charge card,Trouble using your card,Can't use card to make purchases,. Billing Error amount of {$2400.00}. One of t...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NJ,07111,NaN,Consent provided,Web,2023-12-05,Closed with explanation,True,N/A,7878793,92
176445,2023-11-20,Credit card,General-purpose credit card or charge card,Trouble using your card,Can't use card to make purchases,. Billing Error amount of {$2400.00}. One of t...,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NJ,07111,NaN,Consent provided,Web,2023-11-20,Closed with explanation,True,N/A,7879166,92
64806,2023-07-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IL,60651,NaN,Consent provided,Web,2023-07-18,Closed with non-monetary relief,True,N/A,7276305,28
66887,2023-07-17,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,NaN,"EQUIFAX, INC.",TX,782XX,NaN,Consent provided,Web,2023-07-17,Closed with non-monetary relief,True,N/A,7261543,28
100060,2023-10-05,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,NaN,"EQUIFAX, INC.",IL,60517,NaN,Consent provided,Web,2023-10-05,Closed with non-monetary relief,True,N/A,7642120,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245450,2024-03-28,Checking or savings account,Checking account,Opening an account,Unable to open an account,unable to open checking account,NaN,JPMORGAN CHASE & CO.,OH,45044,NaN,Consent provided,Web,2024-03-28,Closed with explanation,True,N/A,8649767,5
151280,2020-11-15,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,unauthorized online purchase was made resultin...,Company has responded to the consumer and the ...,SYNCHRONY FINANCIAL,CA,91770,NaN,Consent provided,Web,2020-11-15,Closed with explanation,True,N/A,3955163,127
282443,2020-11-15,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,unauthorized online purchase was made resultin...,NaN,JPMORGAN CHASE & CO.,CA,91770,NaN,Consent provided,Web,2020-11-15,Closed with explanation,True,N/A,3955162,127
26035,2020-11-19,Checking or savings account,CD (Certificate of Deposit),Closing an account,Funds not received from closed account,under cares act this retirement account was al...,Company has responded to the consumer and the ...,ALLY FINANCIAL INC.,CA,XXXXX,Servicemember,Consent provided,Web,2020-11-19,Closed with explanation,True,N/A,3962772,55


In [35]:
# Group by complaint narrative and count distinct company responses
duplicate_responses = (
    df_pandas
    .groupby('Consumer complaint narrative')['Company response to consumer']
    .nunique()
    .reset_index(name='num_unique_responses')
)

# Keep narratives that have more than one distinct response
duplicate_responses = duplicate_responses[
    duplicate_responses['num_unique_responses'] > 1
]

print(f'Found {len(duplicate_responses):,} narratives that have duplicates with different company responses.')
duplicate_responses.head()

Found 1,271 narratives that have duplicates with different company responses.


,Consumer complaint narrative,num_unique_responses
1551,. See the attached documents. I want the burea...,2
2881,1.I have never had an account with this compan...,2
2893,1.I have never had an account with this compan...,2
3048,15 USC 1692-c If a consumer notifies a debt co...,2
4342,A account was open without my consent in is st...,2


In [36]:
# Filter the df to narratives with multiple responses
conflicting_narratives = duplicate_responses['Consumer complaint narrative']

conflicting_records = (
    df_pandas[
        df_pandas['Consumer complaint narrative'].isin(conflicting_narratives)
    ]
    .sort_values(
        ['Consumer complaint narrative', 'Company response to consumer']
    )
)

conflicting_records.head(20)

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID,word_count
428610,2023-10-05,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,IL,60517,NaN,Consent provided,Web,2023-10-05,Closed with explanation,True,N/A,7642010,28
64806,2023-07-18,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IL,60651,NaN,Consent provided,Web,2023-07-18,Closed with non-monetary relief,True,N/A,7276305,28
66887,2023-07-17,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,NaN,"EQUIFAX, INC.",TX,782XX,NaN,Consent provided,Web,2023-07-17,Closed with non-monetary relief,True,N/A,7261543,28
100060,2023-10-05,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,NaN,"EQUIFAX, INC.",IL,60517,NaN,Consent provided,Web,2023-10-05,Closed with non-monetary relief,True,N/A,7642120,28
119243,2023-10-05,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",IL,60517,NaN,Consent provided,Web,2023-10-05,Closed with non-monetary relief,True,N/A,7642122,28
446940,2023-07-17,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a credit reporting company's inve...,Was not notified of investigation status or re...,. See the attached documents. I want the burea...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,782XX,NaN,Consent provided,Web,2023-07-17,Closed with non-monetary relief,True,N/A,7261913,28
50285,2023-03-28,Credit card or prepaid card,General-purpose credit card or charge card,Incorrect information on your report,Information belongs to someone else,1.I have never had an account with this compan...,NaN,"Bread Financial Holdings, Inc.",TN,XXXXX,NaN,Consent provided,Web,2023-03-28,Closed with explanation,True,N/A,6753972,433
429456,2023-03-16,Credit card or prepaid card,General-purpose credit card or charge card,Incorrect information on your report,Information belongs to someone else,1.I have never had an account with this compan...,NaN,"Bread Financial Holdings, Inc.",MI,48221,NaN,Consent provided,Web,2023-03-16,Closed with non-monetary relief,True,N/A,6705274,433
220624,2023-03-16,Credit card or prepaid card,General-purpose credit card or charge card,Incorrect information on your report,Information belongs to someone else,1.I have never had an account with this compan...,Company has responded to the consumer and the ...,SYNCHRONY FINANCIAL,TN,38018,NaN,Consent provided,Web,2023-03-16,Closed with explanation,True,N/A,6698084,433
58771,2023-02-07,Credit card or prepaid card,General-purpose credit card or charge card,Incorrect information on your report,Information belongs to someone else,1.I have never had an account with this compan...,Company has responded to the consumer and the ...,SYNCHRONY FINANCIAL,TN,37130,NaN,Consent p

In [37]:
# TO DO - Remove or normalize CFPB redactions:

    # df_with_counts['narrative_clean_for_model'] = (
    #     df_with_counts['narrative_clean_for_model']
    #     .fillna('')
    #     .str.replace(r'XX/XX/XXXX', ' REDACTED_DATE ', regex=True)
    #     .str.replace(r'X{2,}', ' REDACTED ', regex=True)
    # )

In [38]:
# Notes for Feature Engineering

# 1. Use a better tokenizer and be wary of how contractions are split. Consider removing single-character tokens:
    # didn't  → did + t
    # it's    → it + s
    # that's  → that + s

# 2. Consider using np.log1p(word_count) as a model feature rather than raw word_count.
    # Rationale:
    # - less sensitive to extreme outliers
    # - more symmetric distribution
    # - often better behaved in linear models
    # - easier to interpret statistically


## UNSUPERVISED ONLY (LDA/KMeans) ##
# 1. Consider truncating inputs above a MAX_CHARS threshold

    # MAX_CHARS = 10000
    #
    # df_with_counts['narrative_clean_for_model'] = (
    #     df_with_counts['Consumer complaint narrative']
    #     .fillna('')
    #     .str.slice(0, MAX_CHARS)
    # )

# 2. Determine strategy for min_chars

In [48]:
# Name the notebook "02_cleaning" keep in development


In [ ]:
# Save to parquet file